# 02 — Pré-processamento

Preparação das variáveis cadastrais e construção do alvo por cliente a partir do histórico de pagamentos.

In [8]:
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
RAW = Path("..") / "data" / "raw"
APPLICATION_FILE = RAW / "application_record.csv"
CREDIT_FILE = RAW / "credit_record.csv"
PROCESSED = Path("..") / "data" / "processed"
TARGET = "TARGET"

pd.set_option("display.max_columns", None)

In [9]:
app_df = pd.read_csv(APPLICATION_FILE)
credit_df = pd.read_csv(CREDIT_FILE)
print(f"Aplicações: {app_df.shape}")
print(f"Histórico de crédito: {credit_df.shape}")

Aplicações: (438557, 18)
Histórico de crédito: (1048575, 3)


## 1. Conferindo dados faltantes e repetidos

A base de histórico tem várias linhas para cada cliente porque registra vários meses. Já um ID repetido na base cadastral pode indicar dois cadastros para a mesma pessoa; como não sei qual usar, vou retirar esses IDs repetidos.

In [10]:
nulos = app_df.isna().sum()
display(pd.DataFrame({"nulos": nulos, "pct": (nulos / len(app_df) * 100).round(2)}).query("nulos > 0").sort_values("nulos", ascending=False))
print("IDs de aplicação duplicados:", app_df["ID"].duplicated().sum())
print("Ocupações ausentes:", app_df["OCCUPATION_TYPE"].isna().sum())

,nulos,pct
OCCUPATION_TYPE,134203,30.6


IDs de aplicação duplicados: 47
Ocupações ausentes: 134203


**Decisão:** vou retirar os cadastros com ID repetido para não escolher um deles sem saber qual está correto. Quando a ocupação estiver vazia, vou usar `Outros`. O tempo de emprego vazio será preenchido durante o treino do modelo, sem consultar os dados de teste.

## 2. Como defini as classes do alvo

Marquei como `TARGET = 1` o cliente que teve atraso de 60 dias ou mais em algum mês (status 2, 3, 4 ou 5). `TARGET = 0` significa que não apareceu atraso de 60 dias ou mais no histórico observado. Essa classe ainda pode incluir clientes que tiveram atrasos menores; por isso, não quer dizer que nunca atrasaram.

In [11]:
credit_df["MAU_PAGADOR"] = credit_df["STATUS"].astype(str).isin(["2", "3", "4", "5"])
target_df = credit_df.groupby("ID", as_index=False)["MAU_PAGADOR"].max()
target_df["TARGET"] = target_df.pop("MAU_PAGADOR").astype(int)

app_df = app_df.loc[~app_df["ID"].duplicated(keep=False)].copy()
df = app_df.merge(target_df, on="ID", how="inner")
print("Distribuição do alvo:")
display(df["TARGET"].value_counts().rename_axis("TARGET").to_frame("clientes"))
print("Base após o merge:", df.shape)

Distribuição do alvo:


,clientes
TARGET,
0,35841
1,616


Base após o merge: (36457, 19)


## 3. Colocando números em escalas parecidas

Não vou mudar a escala nesta etapa. O notebook 03 faz isso depois de separar os dados de treino e de teste. Assim, o modelo não usa por engano informações do teste enquanto aprende.

## 4. Ajustes nas variáveis

Transformo idade e tempo de emprego de dias para anos. O valor 365243 não é um tempo de emprego real, então vira um valor vazio. Preencho ocupações vazias e pouco comuns com `Outros`. Também limito filhos a 5 e tamanho da família a 7 para reduzir o efeito de casos raros. Retiro o indicador de celular porque ele tem o mesmo valor para todos os clientes.

In [12]:
df["OCCUPATION_TYPE"] = df["OCCUPATION_TYPE"].fillna("Outros")
df["DAYS_BIRTH"] = df["DAYS_BIRTH"].abs() / 365.25
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan).abs() / 365.25
df["CNT_CHILDREN"] = df["CNT_CHILDREN"].clip(upper=5)
df["CNT_FAM_MEMBERS"] = df["CNT_FAM_MEMBERS"].clip(upper=7)

frequencia_ocupacao = df["OCCUPATION_TYPE"].value_counts(normalize=True)
ocupacoes_principais = frequencia_ocupacao[frequencia_ocupacao >= 0.03].index
df["OCCUPATION_TYPE"] = df["OCCUPATION_TYPE"].where(
    df["OCCUPATION_TYPE"].isin(ocupacoes_principais), "Outros"
)
if df["FLAG_MOBIL"].nunique(dropna=False) == 1:
    df = df.drop(columns="FLAG_MOBIL")

display(df.head())
print("Nulos restantes:", int(df.isna().sum().sum()))

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,TARGET
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,32.867899,12.435318,1,0,0,Outros,2.0,0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,32.867899,12.435318,1,0,0,Outros,2.0,0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,58.792608,3.104723,0,0,0,Outros,2.0,0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,52.320329,8.353183,0,1,1,Sales staff,1.0,0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,52.320329,8.353183,0,1,1,Sales staff,1.0,0


Nulos restantes: 6135


## 5. Salvando a base preparada

Salvo as categorias com seus nomes para facilitar a leitura. No notebook 03, o computador transforma essas categorias em números e preenche os valores vazios enquanto treina cada modelo.

In [13]:
PROCESSED.mkdir(parents=True, exist_ok=True)
output_path = PROCESSED / "dataset_tratado.csv"
df.to_csv(output_path, index=False)
print(f"Dataset salvo em {output_path} ({df.shape[0]} linhas, {df.shape[1]} colunas).")

Dataset salvo em ..\data\processed\dataset_tratado.csv (36457 linhas, 18 colunas).
